# Pillar3k — DECISIVENESS-weighted crisis distillation

**The story so far (all measured):**
- pillar3g failed: V14 corpus was 88% flat selfplay → de-peaked pillar3f.
- pillar3k v1 (70/30 mix, T=0.7) ALSO failed — regressed everything (val rose from ep1, the de-peak signature). Why: the corpus is dominated by **flat quiet/recovered positions**; the decisive escapes are only 15–22%. **No temperature fixes it** — even T=0.25 only reaches top-share 0.30, still below pillar3f's 0.40. A strong base can't be sharpened by a flat corpus.

**The fix (this notebook): weight each state's loss by its DECISIVENESS** = `(visit top-share)**power`, mean-1 normalized.
- **Decisive escape-or-die states** (top-share ~0.6–0.9) → high weight → **drive the gradient** (teach the floor-saving escapes).
- **Flat quiet states** (top-share ~0.2, "many moves okay") → near-zero weight (≈64× less at power 3) → **can't flatten the policy.**

This separates the escape signal from the flat quiet bulk that sank every uniform full-corpus attempt. The quiet positions still sit in the corpus (broad coverage) but no longer de-peak; the escapes — the actual floor lever — finally dominate learning.

**Corpus:** `v14_rev3.pt` = all crisis (1.59M — incl. crisis_v16_1600 @1600/2400) + 30% selfplay (0.67M) = 2,259,096 states. **Warm-start pillar3f.**

**Sweep (two knobs):** primary `DECISIVENESS=3.0, T=0.7`. Then DECISIVENESS ∈ {2,4} and T ∈ {0.5,1.0}. Change config, re-run.

**Eval by GAMEPLAY, both floor AND quiet-play.** Bar = pillar3f (775000–775499): **mean 35,810 / P50 27,434 / P10 4,546 / <1000 1.2%.** **Now val should FALL** (learning the escapes) rather than rise from ep1 — if it still rises, the @600 crisis escapes are too under-resolved and we need more v16 @4800.

**Upload to Drive (`MyDrive/alphatrain/`):** `colorlines_pillar3d_v2.tar.gz` (REBUILT — has the decisiveness loss), `v14_rev3.pt.gz`, `pillar3f.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar3f.pt', '/content/alphatrain/data/pillar3f.pt')
print(f'Warm-start (pillar3f): {os.path.getsize("/content/alphatrain/data/pillar3f.pt")/1e6:.0f} MB')

t0=time.time()
!cp {DRIVE}/v14_rev3.pt.gz /content/v14_rev3.pt.gz
gz=os.path.getsize('/content/v14_rev3.pt.gz'); print(f'.gz: {gz:,} bytes')
assert gz == 203_353_661, f'.gz truncated! got {gz}; re-upload v14_rev3.pt.gz'
!gunzip -t /content/v14_rev3.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/v14_rev3.pt.gz > /content/alphatrain/data/v14_rev3.pt
pt=os.path.getsize('/content/alphatrain/data/v14_rev3.pt')
assert pt == 865_239_305, f'.pt size wrong! got {pt}'
print(f'v14_rev3 tensor: {pt/1e9:.2f} GB, 2,259,096 states ({time.time()-t0:.0f}s)')
!rm /content/v14_rev3.pt.gz
!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g=torch.cuda.get_device_properties(0)
    print(f'GPU {torch.cuda.get_device_name(0)} | {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG — two knobs =====
DECISIVENESS = 3.0           # (top-share)**power. 0=uniform (failed). Sweep 2 / 3 / 4.
T            = 0.7           # target sharpening. With decisiveness on, keep gentle. Sweep 0.5 / 1.0.
RUN          = 'pillar3k_dw3_T0.7'   # rename per (DECISIVENESS, T)
EPOCHS       = 28           # 3k-v1 still improving at ep16 -> train longer
print(f'RUN={RUN}  DECISIVENESS={DECISIVENESS}  T={T}  EPOCHS={EPOCHS}')

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/v14_rev3.pt \
    --amp --compile \
    --resume alphatrain/data/pillar3f.pt --warm-start \
    --epochs {EPOCHS} --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --target-temperature {T} --decisiveness-power {DECISIVENESS} \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/checkpoints/{RUN} 2>&1 | tee /content/{RUN}_train.log
# val should FALL now (learning escapes). Rising from ep1 = escapes too weak -> need v16 @4800.

In [ ]:
# Copy every epoch to Drive (eval gameplay per-epoch, not val).
import shutil, os, glob
DRIVE='/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst=f'{DRIVE}/{RUN}_{os.path.basename(f)}'; shutil.copy(f,dst); print('Saved', dst)
for f in ['best.pt','latest.pt']:
    s=f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(s): shutil.copy(s,f'{DRIVE}/{RUN}_{f}'); print('Saved', f'{DRIVE}/{RUN}_{f}')

## Eval — per epoch, BOTH floor and quiet-play

On M5 for each epoch ckpt:
```
# 1) FLOOR (did the weighted escapes lift it?)
python -m scripts.eval_policy --model <ckpt> --device cuda --batch 1024 \
    --seed-start 775000 --seed-end 775999
# 2) QUIET-PLAY (decisiveness should protect it — flat states had ~0 weight)
python scripts/normal_play_drift.py --base alphatrain/data/pillar3f.pt --corrected <ckpt> --n 400
```
Bar = pillar3f (775000–775499): **mean 35,810 / P50 27,434 / P10 4,546 / <1000 1.2%**.
- **Floor ≥ bar AND quiet-play ~unchanged** → the decisiveness weighting worked: escapes installed, quiet play protected. Pick best epoch; confirm on 5k (775000–779999). Then iterate (re-mine 4800 crisis on it).
- **Floor up, quiet-play drifts** → raise DECISIVENESS (4) to suppress the quiet states harder.
- **Floor still flat + val rose from ep1** → the @600 crisis escapes are too under-resolved; the lever needs more v16 @4800 (the only corpus with genuinely-peaked escapes). Keep generating v16.